# MCLDNN — Experiment Comparison: 5-Class vs 4-Class Ablation

This notebook loads **pre-trained weights** for both experiments and generates  
side-by-side comparison plots and summary tables.

### Required Kaggle Inputs
- `rml2016-5class`      → `RML2016.10a_5class.pkl`
- `rml2016-4class`      → `RML2016.10a_4class.pkl`
- `rml2016-checkpoints` → both `.weights.h5` files
- `amr-repo`            → repository source code (or git clone)


In [ ]:
# ── CELL 1: Setup ─────────────────────────────────────────────────────────────
import sys, os
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/AMR.git'  # <-- update
REPO_DIR    = '/kaggle/working/AMR'
!git clone --depth 1 {GITHUB_REPO} {REPO_DIR}
sys.path.insert(0, REPO_DIR)
%cd {REPO_DIR}
!pip install -q pyyaml

import tensorflow as tf, keras
print(f'TF={tf.__version__}  Keras={keras.__version__}')

In [ ]:
# ── CELL 2: Point to weights and dataset paths ────────────────────────────────

# Trained weights (from Kaggle dataset or GitHub Release)
WEIGHTS_5CLASS = '/kaggle/input/rml2016-checkpoints/mcldnn_5class_baseline_v1.0_best.weights.h5'
WEIGHTS_4CLASS = '/kaggle/input/rml2016-checkpoints/mcldnn_4class_ablation_v1.0_best.weights.h5'

# Dataset paths
DATA_5CLASS = '/kaggle/input/rml2016-5class/RML2016.10a_5class.pkl'
DATA_4CLASS = '/kaggle/input/rml2016-4class/RML2016.10a_4class.pkl'

# Output directory for comparison figures
SAVE_DIR = '/kaggle/working/comparison_5vs4'
os.makedirs(SAVE_DIR, exist_ok=True)

print('Paths configured.')

In [ ]:
# ── CELL 3: Evaluate both experiments ────────────────────────────────────────
import yaml

def patch_config(config_path, data_path, out_dir):
    with open(config_path) as f:
        cfg = yaml.safe_load(f)
    cfg['dataset']['path']   = data_path
    cfg['output']['figure_dir'] = os.path.join(out_dir, 'figures')
    cfg['output']['result_dir'] = os.path.join(out_dir, 'results')
    patched_path = f'/kaggle/working/{os.path.basename(config_path)}'
    with open(patched_path, 'w') as f:
        yaml.dump(cfg, f)
    return patched_path

cfg5_path = patch_config('configs/exp_5class_baseline.yaml',
                         DATA_5CLASS, os.path.join(SAVE_DIR, '5class'))
cfg4_path = patch_config('configs/exp_4class_ablation.yaml',
                         DATA_4CLASS, os.path.join(SAVE_DIR, '4class'))

# Run evaluation and comparison
!python src/evaluate.py \
    --compare \
    --configs  {cfg5_path} {cfg4_path} \
    --weights  {WEIGHTS_5CLASS} {WEIGHTS_4CLASS} \
    --labels   "5-class baseline" "4-class ablation (no BPSK)" \
    --save_dir {SAVE_DIR}

In [ ]:
# ── CELL 4: Display comparison results ───────────────────────────────────────
from IPython.display import Image, display
import glob, csv

# Main comparison figure
cmp_fig = os.path.join(SAVE_DIR, 'comparison_acc_vs_snr.png')
if os.path.exists(cmp_fig):
    display(Image(cmp_fig, width=800))

# Summary table
summary_csv = os.path.join(SAVE_DIR, 'comparison_summary.csv')
if os.path.exists(summary_csv):
    import pandas as pd
    df = pd.read_csv(summary_csv)
    display(df)

In [ ]:
# ── CELL 5: Individual confusion matrices ─────────────────────────────────────
from IPython.display import Image, display
import glob

print('=== 5-class baseline: all-SNR confusion matrix ===')
fig5 = os.path.join(SAVE_DIR, '5class/figures/confusion_all_snrs.png')
if os.path.exists(fig5): display(Image(fig5, width=500))

print('=== 4-class ablation: all-SNR confusion matrix ===')
fig4 = os.path.join(SAVE_DIR, '4class/figures/confusion_all_snrs.png')
if os.path.exists(fig4): display(Image(fig4, width=500))